# 02 · Train the pocket-conditioned equivariant diffusion model

Learns `p(ligand | pocket)` — joint diffusion over ligand **coordinates** and **atom types**,
denoised by an E(n)-equivariant GNN with the pocket as frozen context.

**Input** → the output of notebook 01 (`graphs/` + `src/molgen`). **Output** → checkpoints in
`/kaggle/working/`, resumable across sessions.

### The model
Forward process, per ligand atom, on `z = x / coord_scale` and one-hot types `h / type_scale`:

$$q(z_t \mid z_0) = \mathcal{N}\!\left(\alpha_t z_0,\; \sigma_t^2 I\right)$$

The network predicts the **denoised** $\hat x_0$ and training minimises
$\lVert \hat x_0 - x_0 \rVert^2$ over coordinates and types, plus a cross-entropy term on
element identity.

**Why $x_0$ and not $\epsilon$.** With noise prediction the reverse mean is
$z_t/\alpha_{t|s} - (\sigma^2_{t|s}/\alpha_{t|s}\sigma_t)\,\hat\epsilon$ — it **divides** $z$
by $\alpha_{t|s} < 1$ every step, amplifying it by $\alpha_0/\alpha_T \approx 316\times$ over
the trajectory, and only a perfect $\hat\epsilon$ cancels that. Here the error does not merely
degrade samples, it runs away: once an atom drifts past the 8 Å cross-edge cutoff the pocket
context vanishes from its neighbourhood, the network predicts almost nothing, and the rest of
the trajectory is pure amplification. *Measured:* a model overfit to 8 complexes put its
samples **331 Å** from the pocket (untrained: 472 Å) with the loss already down 78 %.

Predicting $x_0$ gives $\mu = c_t z_t + c_0 \hat x_0$ with $c_t, c_0 \ge 0$ and
$c_t + c_0 \approx 1$ — an interpolation, so nothing is amplified, and a bad prediction
collapses samples toward the pocket centroid where the context still reaches them. Same
posterior, numerically stable half. This is what TargetDiff does for the same task.

**Why equivariance.** Coordinate updates are built only from *relative* positions scaled by
scalar messages:

$$x_i \leftarrow x_i + \sum_j \frac{x_i - x_j}{d_{ij} + 1}\, \phi_x(m_{ij})$$

so rotating the input rotates the output exactly, by construction. A model on raw coordinates
would have to learn that symmetry from data — unaffordable at this dataset size.

**Why the pocket is frozen.** Pocket atoms join message passing but their coordinates never
move, and the complex was centred on the pocket centroid during preprocessing. That handles
translation, so no zero-centre-of-mass subspace projection is needed and every radius cutoff
keeps its meaning in Ångström.

In [ ]:
# --- locate the molgen package ---------------------------------------------
# On Kaggle, make `molgen` importable in one of these ways:
#   1. Add the "molgen-src" Dataset (upload this repo's `src/` folder), or
#   2. Add the output of notebook 01 (it ships a copy of `src/molgen`), or
#   3. Run the notebook from a clone of the repo.
import glob
import subprocess
import sys
from pathlib import Path


def find_molgen():
    patterns = [
        "/kaggle/input/*/src", "/kaggle/input/*", "/kaggle/input/*/*",
        "/kaggle/working/src", "./src", "../src", ".", "..",
    ]
    for pattern in patterns:
        for candidate in sorted(glob.glob(pattern)):
            if (Path(candidate) / "molgen" / "__init__.py").is_file():
                return str(Path(candidate).resolve())
    return None


MOLGEN_SRC = find_molgen()
if MOLGEN_SRC is None:
    raise FileNotFoundError(
        "Could not find the `molgen` package.\n"
        "Attach the molgen-src dataset (the repo's src/ folder, containing molgen/), "
        "or the output of notebook 01, then re-run this cell."
    )
if MOLGEN_SRC not in sys.path:
    sys.path.insert(0, MOLGEN_SRC)
print("molgen source:", MOLGEN_SRC)

In [ ]:
# --- dependencies -----------------------------------------------------------
# torch_geometric is required. We deliberately avoid torch_scatter/torch_cluster
# (they need wheels matched to the exact torch build and are the usual cause of
# Kaggle install failures) -- molgen uses scipy KD-trees and native torch ops.
# rdkit is optional: without it you lose SMILES/QED/SA but nothing else.
def ensure(module: str, package: str | None = None) -> bool:
    try:
        __import__(module)
        return True
    except ImportError:
        pkg = package or module
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    try:
        __import__(module)
        return True
    except ImportError:
        print(f"  !! {module} unavailable")
        return False


HAS_PYG = ensure("torch_geometric")
HAS_RDKIT = ensure("rdkit")
assert HAS_PYG, "torch_geometric is required"
print("rdkit:", HAS_RDKIT)

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import molgen
from molgen import (
    affinity, chem, dataset, diffusion, egnn, evaluate, graphs, pdbio, pockets, sources,
)
from molgen.train_utils import (
    EMA, History, TimeBudget, count_parameters, device_info, grad_norm,
    load_checkpoint, save_checkpoint, set_seed, warmup_cosine_lambda,
)

warnings.filterwarnings("ignore", category=UserWarning)
set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("molgen", molgen.__version__, "| device:", device)
print(device_info())

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

## 1 · Load the dataset and its metadata

`coord_scale` comes from the dataset, not a guess — see notebook 01 §6.

In [ ]:
GRAPH_DIR = next(
    (p for p in [Path(g) for g in glob.glob("/kaggle/input/*/graphs")] + [WORK / "graphs"]
     if (p / "train_manifest.json").exists()),
    None,
)
assert GRAPH_DIR is not None, "could not find graphs/ — attach the notebook-01 output"
print("graphs:", GRAPH_DIR)

meta = json.loads((GRAPH_DIR / "dataset_meta.json").read_text())
build = meta["build_config"]
print(json.dumps(meta, indent=2))

train_ds = dataset.ShardedComplexDataset(GRAPH_DIR, "train", cache_shards=2,
                                         pocket_cutoff=build["pocket_cutoff"],
                                         inter_cutoff=build["inter_cutoff"])
val_ds = dataset.ShardedComplexDataset(GRAPH_DIR, "val", cache_shards=1,
                                       pocket_cutoff=build["pocket_cutoff"],
                                       inter_cutoff=build["inter_cutoff"])
size_prior = dataset.SizePrior.load(GRAPH_DIR / "size_prior.json")
print(f"\ntrain {len(train_ds):,} | val {len(val_ds):,}")

## 2 · Configuration

Defaults are sized for one Kaggle P100/T4 session. The knobs that matter most:

| knob | effect |
|---|---|
| `timesteps` | 500 is plenty for molecules; 1000 doubles sampling cost for little gain |
| `hidden` / `num_layers` | 128×6 ≈ 1.4 M params, fits comfortably; 256×9 if you have the hours |
| `cross_cutoff` | how far ligand atoms see the pocket. 8 Å covers the first two shells |
| `type_loss_weight` / `type_ce_weight` | ↑ favours correct elements, ↓ favours correct geometry |
| `parameterization` | leave at `"x0"`. `"eps"` is there for comparison and is unstable here |
| `batch_size` | pocket graphs are the memory cost — 16 is safe at 600 pocket atoms |

In [ ]:
DCFG = diffusion.DiffusionConfig(
    timesteps=500,
    schedule="polynomial",          # decays slowly at small t: fine geometry is set late
    coord_scale=meta["coord_scale"],
    type_scale=4.0,                 # EDM convention; keeps type signal alive under noise
    type_loss_weight=1.0,
    type_ce_weight=0.3,             # auxiliary CE sharpens element identity
    parameterization="x0",          # see the note above — do not switch to "eps"
    clip_x0=4.0,                    # normalised units from the pocket centroid
    hidden=128,
    num_layers=6,
    cross_cutoff=8.0,
    cross_max_neighbors=32,
    attention=True,
    ligand_type_dim=meta["feature_dims"]["ligand_type_dim"],
    pocket_feature_dim=meta["feature_dims"]["pocket_feature_dim"],
)

TRAIN = dict(
    epochs=200,
    batch_size=16,
    lr=2e-4,
    weight_decay=1e-12,
    grad_clip=1.0,
    warmup_steps=1000,
    ema_decay=0.999,
    num_workers=2,
    val_every=1,
    log_every=100,
    max_steps_per_epoch=None,       # cap for quick experiments
    session_hours=8.5,              # Kaggle kills at 9 h
)

CKPT = WORK / "diffusion_last.pt"
BEST = WORK / "diffusion_best.pt"
RESUME = CKPT if CKPT.exists() else None     # set to a previous session's checkpoint
print(DCFG)

## 3 · Build the model and **verify the symmetry numerically**

Run this every time. A broken equivariance is completely silent — the model still trains, just
badly — so it has to be asserted, not assumed. Rotating the complex must rotate `eps_x` by the
same matrix and leave the type prediction unchanged.

In [ ]:
model = diffusion.PocketLigandDiffusion(DCFG).to(device)
print(f"{count_parameters(model):,} trainable parameters")

check_batch = diffusion.build_pocket_batch([train_ds[i] for i in range(4)], keep_ligand=True)
report = egnn.check_equivariance(model.net, check_batch, device="cpu")
print(json.dumps(report, indent=2))
assert report["passed"], "E(3) equivariance broken — do not train this"
print(f"\nall four symmetry checks hold to {report['worst_error']:.1e}")
model = model.to(device)

In [ ]:
train_loader = dataset.make_loader(train_ds, batch_size=TRAIN["batch_size"], shuffle=True,
                                   num_workers=TRAIN["num_workers"], drop_last=True)
val_loader = dataset.make_loader(val_ds, batch_size=TRAIN["batch_size"], shuffle=False,
                                 num_workers=TRAIN["num_workers"])

opt = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"], weight_decay=TRAIN["weight_decay"])
steps_per_epoch = TRAIN["max_steps_per_epoch"] or len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, warmup_cosine_lambda(TRAIN["warmup_steps"], TRAIN["epochs"] * steps_per_epoch)
)
ema = EMA(model, decay=TRAIN["ema_decay"])
history = History(WORK / "diffusion_history.jsonl")
budget = TimeBudget(limit_s=TRAIN["session_hours"] * 3600)

start_epoch, global_step, best_val = 0, 0, float("inf")
if RESUME is not None:
    ck = load_checkpoint(RESUME, model, opt, sched, ema, map_location=device)
    start_epoch, global_step = ck["epoch"] + 1, ck["step"]
    best_val = ck.get("extra", {}).get("best_val", float("inf"))
    print(f"resumed from {RESUME} at epoch {start_epoch}, step {global_step:,}")

print(f"{steps_per_epoch:,} steps/epoch | {budget}")

## 4 · Validation loss, made comparable

The training loss samples `t` at random, so epoch-to-epoch differences are mostly noise in the
`t` draw. Validation therefore uses a **fixed seed**, which makes the curve actually readable.

In [ ]:
@torch.no_grad()
def validate(net, loader, max_batches=50, seed=1234):
    net.eval()
    totals, n = {}, 0
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        torch.manual_seed(seed + i)          # same noise draw every evaluation
        out = net(batch.to(device))
        for k, v in out.items():
            totals[k] = totals.get(k, 0.0) + float(v)
        n += 1
    net.train()
    return {k: v / max(n, 1) for k, v in totals.items()}

## 5 · Train

In [ ]:
model.train()
stop = False
for epoch in range(start_epoch, TRAIN["epochs"]):
    t_epoch = time.time()
    run, seen = {}, 0

    for i, batch in enumerate(train_loader):
        if TRAIN["max_steps_per_epoch"] and i >= TRAIN["max_steps_per_epoch"]:
            break
        out = model(batch.to(device))
        opt.zero_grad(set_to_none=True)
        out["loss"].backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN["grad_clip"])
        opt.step(); sched.step(); ema.update(model)

        global_step += 1
        seen += 1
        for k, v in out.items():
            run[k] = run.get(k, 0.0) + float(v)

        if global_step % TRAIN["log_every"] == 0:
            avg = {k: v / seen for k, v in run.items()}
            history.log(step=global_step, epoch=epoch, split="train", **avg,
                        lr=sched.get_last_lr()[0], grad_norm=float(gnorm))
            print(f"e{epoch:>3} s{global_step:>7,} | loss {avg['loss']:.4f} "
                  f"(x {avg['loss_x']:.4f}  h {avg['loss_h']:.4f}"
                  f"{'  acc %.2f' % avg['type_acc'] if 'type_acc' in avg else ''}) | "
                  f"lr {sched.get_last_lr()[0]:.2e} | gn {float(gnorm):.2f}", flush=True)

        if budget.expired():
            print(f"\ntime budget reached ({budget}) — checkpointing and stopping")
            stop = True
            break

    if epoch % TRAIN["val_every"] == 0 or stop:
        ema.apply_to(model)                   # EMA weights are what we will sample with
        val = validate(model, val_loader)
        ema.restore(model)
        history.log(step=global_step, epoch=epoch, split="val", **{f"val_{k}": v for k, v in val.items()})
        print(f"  -> epoch {epoch} val loss {val['loss']:.4f} "
              f"(x {val['loss_x']:.4f}  h {val['loss_h']:.4f}"
              f"{'  acc %.2f' % val['type_acc'] if 'type_acc' in val else ''})"
              f" | {time.time()-t_epoch:.0f}s | {budget}")
        if val["loss"] < best_val:
            best_val = val["loss"]
            save_checkpoint(BEST, model, opt, sched, ema, epoch, global_step,
                            config=model.config_dict(), extra={"best_val": best_val, "meta": meta})
            print(f"     new best ({best_val:.4f}) -> {BEST.name}")

    save_checkpoint(CKPT, model, opt, sched, ema, epoch, global_step,
                    config=model.config_dict(), extra={"best_val": best_val, "meta": meta})
    if stop:
        break

print(f"\nfinished at epoch {epoch}, step {global_step:,}; best val {best_val:.4f}")

In [ ]:
df = history.to_frame()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
tr = df[df.split == "train"]; va = df[df.split == "val"]
ax[0].plot(tr.step, tr.loss, lw=0.8, label="train")
if len(va): ax[0].plot(va.step, va.val_loss, "o-", label="val")
ax[0].set_title("total loss"); ax[0].set_yscale("log")
ax[1].plot(tr.step, tr.loss_x, lw=0.8, label="coords (x0 MSE)")
ax[1].plot(tr.step, tr.loss_h, lw=0.8, label="types (MSE)")
if "loss_ce" in tr: ax[1].plot(tr.step, tr.loss_ce, lw=0.8, label="types (CE)")
ax[1].set_title("loss terms"); ax[1].set_yscale("log")
if "type_acc" in tr:
    ax[2].plot(tr.step, tr.type_acc, lw=0.8, label="train")
    if len(va) and "val_type_acc" in va: ax[2].plot(va.step, va.val_type_acc, "o-", label="val")
    ax[2].set_title("element accuracy at random t"); ax[2].set_ylim(0, 1)
else:
    ax[2].plot(tr.step, tr.grad_norm, lw=0.6); ax[2].set_title("grad norm"); ax[2].set_yscale("log")
for a in ax: a.set_xlabel("step"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6 · Sample from held-out pockets

The real test of a generator is not its loss. `build_pocket_batch` **strips the crystal
ligand** — generating with the true ligand still in the batch is the easiest way to report
leaked results, so it is made structurally impossible rather than promised.

Expect from a short run: coordinates in the right region but poor validity. Validity above
~80 % with mostly single fragments needs a few hundred epochs.

In [ ]:
N_POCKETS, PER_POCKET, SAMPLE_STEPS = 8, 5, 250

ema.apply_to(model)
model.eval()

rng = np.random.default_rng(0)
pick = rng.choice(len(val_ds), size=min(N_POCKETS, len(val_ds)), replace=False)
records, generated = [], []

for idx in pick:
    truth = val_ds[int(idx)]
    repeats = [truth] * PER_POCKET
    batch = diffusion.build_pocket_batch(repeats, device=device)   # ligand stripped here
    n_atoms = [size_prior.sample(int(truth.n_poc_res), rng) for _ in range(PER_POCKET)]

    out = model.sample(batch, n_atoms=n_atoms, num_steps=SAMPLE_STEPS, eta=1.0)
    poc_pos = truth.poc_pos.numpy()
    # real vdW radii per pocket atom: defaulting everything to carbon
    # under-counts clashes against sulfur and over-counts against oxygen
    poc_el = graphs.pocket_elements_from_features(truth.poc_x)
    for mol in model.to_molecules(out):
        rec = evaluate.evaluate_molecule(mol["elements"], mol["pos"], poc_pos=poc_pos,
                                         poc_elements=poc_el,
                                         index=len(records), pocket_id=truth.complex_id)
        records.append(rec)
        generated.append((truth, mol))

ema.restore(model)
metrics = evaluate.set_metrics(records)
print(json.dumps(metrics, indent=2))

# The check that matters first, before any chemistry: did the molecules land in
# the pocket at all? Compare the generated centroid to the crystal ligand's.
# A number in the tens or hundreds of Angstrom means the reverse process is
# diverging, not that the chemistry needs more epochs.
errs = [float(np.linalg.norm(mol["pos"].mean(0) - truth.lig_pos.numpy().mean(0)))
        for truth, mol in generated]
rg_gen = [float(np.sqrt(((m["pos"] - m["pos"].mean(0)) ** 2).sum(1).mean())) for _, m in generated]
rg_true = [float(np.sqrt(((t.lig_pos.numpy() - t.lig_pos.numpy().mean(0)) ** 2).sum(1).mean()))
           for t, _ in generated]
print(f"\ncentroid offset from the crystal ligand: mean {np.mean(errs):.2f} A, max {np.max(errs):.2f} A")
print(f"radius of gyration: generated {np.mean(rg_gen):.2f} A vs crystal {np.mean(rg_true):.2f} A")
assert np.mean(errs) < 25.0, (
    f"samples are {np.mean(errs):.0f} A from the pocket — the reverse process is diverging, "
    "not under-trained. Check DCFG.parameterization == 'x0' and clip_x0."
)
history.log(step=global_step, epoch=epoch, split="sample", **{f"gen_{k}": v for k, v in metrics.items()
                                                              if isinstance(v, (int, float))})

In [ ]:
tbl = pd.DataFrame([r.row() for r in records])
cols = [c for c in ["pocket_id", "n_atoms", "valid", "n_fragments", "min_dist",
                    "n_clash", "buriedness", "qed", "sa", "smiles"] if c in tbl.columns]
display(tbl[cols].head(20))

# reference: the same statistics for the real ligands of those pockets
real = [evaluate.evaluate_molecule(
            [chem.LIGAND_ELEMENTS[t] for t in val_ds[int(i)].lig_type.tolist()],
            val_ds[int(i)].lig_pos.numpy(), poc_pos=val_ds[int(i)].poc_pos.numpy(),
            poc_elements=graphs.pocket_elements_from_features(val_ds[int(i)].poc_x))
        for i in pick]
print("\ngenerated:", {k: round(v, 3) for k, v in evaluate.set_metrics(records).items() if isinstance(v, float)})
print("crystal  :", {k: round(v, 3) for k, v in evaluate.set_metrics(real).items() if isinstance(v, float)})

In [ ]:
# write the best molecules per pocket back into the crystal frame for PyMOL
best_by_pocket = {}
for rec, (truth, _) in zip(records, generated):
    rec.score = evaluate.composite_score(rec)
    if rec.score > best_by_pocket.get(rec.pocket_id, (None, -1e9))[1]:
        best_by_pocket[rec.pocket_id] = (rec, rec.score, truth)

blocks = 0
for pid, (rec, _, truth) in best_by_pocket.items():
    blocks += evaluate.records_to_sdf([rec], WORK / f"sample_{pid}.sdf",
                                      offset=truth.center.numpy().reshape(3))
print(f"wrote {blocks} molecules as sample_*.sdf (crystal frame — open next to the receptor PDB)")

save_checkpoint(WORK / "diffusion_final.pt", model, ema=ema, epoch=epoch, step=global_step,
                config=model.config_dict(),
                extra={"best_val": best_val, "meta": meta, "sample_metrics": metrics})
print("saved diffusion_final.pt")

---
**Next:** `03_train_binding_affinity.ipynb` for the ranker, then
`04_generate_and_rank.ipynb` for the full protein → ranked molecules pipeline.

**To continue training in a new session:** attach this notebook's output, point `RESUME` at
`diffusion_last.pt`, and re-run. Optimiser, scheduler and EMA state all restore.